In [1]:
import numpy as np

# 1. Загружаем файл
data = np.load('preparing_data/camera_calibration.npz')

# 2. Смотрим, какие массивы внутри
print("Доступные массивы:", data.files) 
# Выведет: ['camera_matrix', 'dist_coeffs', 'rvecs', 'tvecs', 'image_size']

# 3. Извлекаем нужные данные по именам
camera_matrix = data['camera_matrix']
dist_coeffs = data['dist_coeffs']

print("Матрица камеры:\n", camera_matrix)
print("Коэффициенты дисторсии:\n", dist_coeffs)

Доступные массивы: ['camera_matrix', 'dist_coeffs', 'rvecs', 'tvecs', 'image_size']
Матрица камеры:
 [[1.06069264e+03 0.00000000e+00 9.48428187e+02]
 [0.00000000e+00 1.07290517e+03 5.53733829e+02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]
Коэффициенты дисторсии:
 [[-0.11071497]
 [ 0.1140402 ]
 [-0.13679937]
 [ 0.0578019 ]]


In [1]:
import numpy as np
import os

# === НАСТРОЙКИ ===
INPUT_FILE = 'data/camera_calibration.npz'
OUTPUT_FILE = 'data/camera_calibration_qhd.npz'

TARGET_WIDTH = 960   # HD ширина
TARGET_HEIGHT = 540   # HD высота

# === ЗАГРУЗКА И КОНВЕРТАЦИЯ ===
if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"Файл не найден: {INPUT_FILE}")

print(f"📂 Загрузка калибровки: {INPUT_FILE}")
data = np.load(INPUT_FILE)

# Извлекаем исходную матрицу и размер
K_fullhd = data['camera_matrix'].astype(np.float64)
img_size_fullhd = tuple(data['image_size']) if 'image_size' in data.files else None

print(f"   Исходное разрешение: {img_size_fullhd}")
print(f"   Целевое разрешение:  ({TARGET_WIDTH}, {TARGET_HEIGHT})")

# Вычисляем коэффициенты масштабирования
sx = TARGET_WIDTH / img_size_fullhd[0]
sy = TARGET_HEIGHT / img_size_fullhd[1]

# Создаем новую матрицу камеры с масштабированным фокусным и оптическим центром
K_hd = K_fullhd.copy()
K_hd[0, 0] *= sx  # fx
K_hd[1, 1] *= sy  # fy
K_hd[0, 2] *= sx  # cx
K_hd[1, 2] *= sy  # cy

# Дисторсия остается без изменений
dist_coeffs = data['dist_coeffs'].copy()

# Формируем новый размер изображения
new_image_size = np.array([TARGET_WIDTH, TARGET_HEIGHT])

# Сохраняем в новый файл
np.savez(
    OUTPUT_FILE,
    camera_matrix=K_hd,
    dist_coeffs=dist_coeffs,
    image_size=new_image_size,
    rvecs=data.get('rvecs', np.zeros((0,))),
    tvecs=data.get('tvecs', np.zeros((0,)))
)

print(f"\n✅ Калибровка сохранена: {os.path.abspath(OUTPUT_FILE)}")
print(f"   Новая матрица K:\n{K_hd}")
print(f"   Дисторсия: {dist_coeffs.flatten()}")

📂 Загрузка калибровки: data/camera_calibration.npz
   Исходное разрешение: (1920, 1080)
   Целевое разрешение:  (960, 540)

✅ Калибровка сохранена: c:\Users\user\Desktop\регистратор\data\camera_calibration_qhd.npz
   Новая матрица K:
[[530.34632042   0.         474.21409332]
 [  0.         536.45258348 276.86691455]
 [  0.           0.           1.        ]]
   Дисторсия: [-0.11071497  0.1140402  -0.13679937  0.0578019 ]
